# MUStARD++ exploratory analysis

Phase 2 of the sarcasm detection project. This notebook verifies the official **1,202 / 601 / 601** counts, checks that every utterance has (or is logged as missing) a video file, and plots the distributions that later inform splits and ethics notes.

Run `python data/download.py` first so `data/raw/mustard++_text.csv` exists.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from mustard.dataset import attach_media_paths, build_utterance_table, load_raw_csv, verify_dataset
from mustard.eda import generate_eda

sns.set_theme(style="whitegrid")
raw = load_raw_csv()
df = attach_media_paths(build_utterance_table(raw))
report = verify_dataset(df)
report

## Integrity

Paper numbers: 1,202 instances, 601 sarcastic, 601 non-sarcastic. Rows whose `KEY` ends in `_u` are utterances; `_c_*` rows are context and carry empty sarcasm labels. Missing clips are recorded in `has_video` rather than dropped silently.

In [ ]:
assert report["n_utterances"] == 1202, report
assert report["n_sarcastic"] == 601 and report["n_non_sarcastic"] == 601, report
print("Utterances", report["n_utterances"])
print("Videos present", report["n_with_video"], "/ missing", report["n_utterances"] - report["n_with_video"])
print("Shows", report["shows"])

## Class, length, type, emotion, speaker, duration

In [ ]:
df["utt_tokens"] = df["utterance"].str.split().str.len()
df["ctx_tokens"] = df["context"].str.split().str.len()

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
df["sarcasm"].map({0: "Non-sarcastic", 1: "Sarcastic"}).value_counts().plot(kind="bar", ax=axes[0, 0], color=["#5c5346", "#C9961A"], title="Class")
sns.histplot(df["utt_tokens"], ax=axes[0, 1], color="#C9961A").set_title("Utterance tokens")
df["sarcasm_type"].value_counts().plot(kind="bar", ax=axes[0, 2], color="#C9961A", title="Sarcasm type")
df["SHOW"].value_counts().plot(kind="bar", ax=axes[1, 0], color="#C9961A", title="Show")
df["implicit_emotion"].value_counts().head(12).plot(kind="bar", ax=axes[1, 1], color="#C9961A", title="Implicit emotion")
sns.histplot(df["utt_duration_sec"].dropna(), ax=axes[1, 2], color="#C9961A").set_title("Duration (s)")
fig.tight_layout()
plt.show()
df[["utt_tokens", "ctx_tokens", "utt_duration_sec", "valence", "arousal"]].describe()

## Speaker bias

Chandler / Sheldon style sarcasm is over-represented in sitcom corpora. A speaker-independent holdout (see `mustard.splits.speaker_independent_split`) is required so we do not report a speaker-ID system as sarcasm detection.

In [ ]:
top = df["SPEAKER"].value_counts().head(12).index
pd.crosstab(df["SPEAKER"], df["sarcasm"]).loc[top].plot(kind="bar", stacked=True, color=["#5c5346", "#C9961A"], figsize=(10, 4), title="Sarcasm by speaker")
plt.tight_layout()
plt.show()

## Ethics notes for the report

- Acted US sitcom sarcasm, not spontaneous speech.
- English only.
- Laugh tracks may leak into the audio channel.
- Isolated utterances (the text UI) drop the visual/prosodic evidence the annotators had.

Regenerate publication figures with `generate_eda()`:

In [ ]:
summary = generate_eda()
summary